In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.linear_model import Ridge
from scipy.stats import ttest_rel
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import shap

LOGO CROSS-VALIDATION TESTING

In [ ]:
logo = LeaveOneGroupOut()

phys_cv = cross_val_score(
    Ridge(), X_train_phys_grouped, y_train_grouped,
    cv=logo, groups=groups_grouped,
    scoring="neg_mean_absolute_error"
)
esm_cv = cross_val_score(
    Ridge(), X_train_esm_grouped, y_train_grouped,
    cv=logo, groups=groups_grouped,
    scoring="neg_mean_absolute_error"
)
hybrid_cv = cross_val_score(
    Ridge(), X_train_hybrid_grouped, y_train_grouped,
    cv=logo, groups=groups_grouped,
    scoring="neg_mean_absolute_error"
)
print("\nCROSS VALIDATION RESULTS")
print("Hybrid vs Phys p-value:",
      ttest_rel(-hybrid_cv, -phys_cv)[1])
print("Hybrid vs ESM p-value:",
      ttest_rel(-hybrid_cv, -esm_cv)[1])

def ci(scores):
    mean   = np.mean(-scores)
    margin = 1.96 * np.std(-scores) / np.sqrt(len(scores))
    return mean, margin

print("\nCONFIDENCE INTERVALS (MAE)")
print("Phys:  ", ci(phys_cv))
print("ESM:   ", ci(esm_cv))
print("Hybrid:", ci(hybrid_cv))

BASELINE + ABLATION STUDY

In [ ]:
# baseline models
phys_model   = Ridge().fit(X_train_phys,   y_train)
esm_model    = Ridge().fit(X_train_esm,    y_train)
hybrid_model = Ridge().fit(X_train_hybrid, y_train)

phys_preds   = phys_model.predict(X_test_phys)
esm_preds    = esm_model.predict(X_test_esm)
hybrid_preds = hybrid_model.predict(X_test_hybrid)

# shap ablation
explainer   = shap.Explainer(hybrid_model, X_train_hybrid)
shap_values = explainer(X_test_hybrid[:200])
mean_shap   = np.abs(shap_values.values).mean(axis=0)

n_phys  = X_train_phys.shape[1]
top_idx = np.argsort(mean_shap[n_phys:])[-10:] + n_phys   # ESM dims only

X_train_ablated = np.delete(X_train_hybrid, top_idx, axis=1)
X_test_ablated  = np.delete(X_test_hybrid,  top_idx, axis=1)
ablated_model   = Ridge().fit(X_train_ablated, y_train)
ablated_preds   = ablated_model.predict(X_test_ablated)

NEGATIVE CONTROL

In [ ]:
np.random.seed(42)
y_shuffled = np.random.permutation(y_train)

nc_phys   = Ridge().fit(X_train_phys,   y_shuffled).predict(X_test_phys)
nc_esm    = Ridge().fit(X_train_esm,    y_shuffled).predict(X_test_esm)
nc_hybrid = Ridge().fit(X_train_hybrid, y_shuffled).predict(X_test_hybrid)

RESULTS

In [ ]:
ablation_results = pd.DataFrame([
    {"Model": "Physicochemical",
     "MAE": mean_absolute_error(y_test, phys_preds),
     "R2":  r2_score(y_test, phys_preds)},

    {"Model": "ESM Only",
     "MAE": mean_absolute_error(y_test, esm_preds),
     "R2":  r2_score(y_test, esm_preds)},

    {"Model": "Hybrid",
     "MAE": mean_absolute_error(y_test, hybrid_preds),
     "R2":  r2_score(y_test, hybrid_preds)},

    {"Model": "Ablated (top-10 ESM dims removed)",
     "MAE": mean_absolute_error(y_test, ablated_preds),
     "R2":  r2_score(y_test, ablated_preds)},
])
print("\nABLATION RESULTS")
display(ablation_results)

In [ ]:
negative_control_results = pd.DataFrame([
    {"Model": "Physicochemical (shuffled)",
     "MAE": mean_absolute_error(y_test, nc_phys),
     "R2":  r2_score(y_test, nc_phys)},

    {"Model": "ESM (shuffled)",
     "MAE": mean_absolute_error(y_test, nc_esm),
     "R2":  r2_score(y_test, nc_esm)},

    {"Model": "Hybrid (shuffled)",
     "MAE": mean_absolute_error(y_test, nc_hybrid),
     "R2":  r2_score(y_test, nc_hybrid)},
])
print("\nNEGATIVE CONTROL RESULTS")
display(negative_control_results)

SHAP FEATURE REMOVAL VALIDATION

In [ ]:
# mean SHAP importance
mean_shap = np.abs(shap_values.values).mean(axis=0)
top_features = np.argsort(mean_shap[n_phys:])[-10:] + n_phys

# remove top features
X_train_removed = np.delete(X_train_hybrid, top_features, axis=1)
X_test_removed  = np.delete(X_test_hybrid,  top_features, axis=1)

# retrain model
removed_model = Ridge().fit(X_train_removed, y_train)
removed_preds = removed_model.predict(X_test_removed)

print("\nSHAP FEATURE REMOVAL RESULTS")
print("Removed Model MAE:", mean_absolute_error(y_test, removed_preds))
print("Removed Model R2:",  r2_score(y_test, removed_preds))

# baseline comparison
baseline_preds = hybrid_model.predict(X_test_hybrid)

print("\nBaseline vs SHAP-removed comparison")
print("Baseline MAE:", mean_absolute_error(y_test, baseline_preds))
print("Removed MAE:",  mean_absolute_error(y_test, removed_preds))